In [19]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END
from typing import Literal


class ContentState(TypedDict):
    text:str
    score:float
    message:str
    category:str


def analyze_content(state:ContentState)->dict:
    print(f"Analyzing Content : {state['text'][:50]}....")
    good_words = ["great", "excellent", "amazing", "helpful", "good"]
    bad_words = ["spam", "scam", "fake", "terrible", "hate"]
    text=state['text'].lower()

    good_count=sum(words in text for words in good_words)
    bad_count=sum(words in text for words in bad_words)
    if good_count+bad_count==0:
        score=0.5
    else:
        score=good_count/(good_count+bad_count)
    print(f"Content Score:{score:.2f}")
    return {"score":score}

def route_content(state:ContentState)->Literal["approve","reject","review"]:
    score=state['score']

    if score>=0.7:
        print(f"Routing to approve")
        return "approve"
    elif score<0.3:
        print(f"Routing to reject")
        return "reject"
    else:
        print(f"Routing to review")
        return "review"

def approve_content(state: ContentState) -> dict:
    """Approve the content."""
    print("✅ Content APPROVED")
    return {
        "category": "approved",
        "message": "Content approved for publication"
    }

def reject_content(state: ContentState) -> dict:
    """Reject the content."""
    print("❌ Content REJECTED")
    return {
        "category": "rejected",
        "message": "Content does not meet quality standards"
    }

def review_content(state: ContentState) -> dict:
    """Flag for manual review."""
    print("⚠️  Content flagged for REVIEW")
    return {
        "category": "review",
        "message": "Content requires manual review"
    }

In [20]:


content_graph=StateGraph(ContentState)
content_graph.add_node("analyze",analyze_content)
content_graph.add_node("approve",approve_content)
content_graph.add_node("reject",reject_content)
content_graph.add_node("review",review_content)

content_graph.add_edge(START,"analyze")

content_graph.add_conditional_edges(
    "analyze",
    route_content
)
content_graph.add_edge("approve",END)
content_graph.add_edge("reject",END)
content_graph.add_edge("review",END)

content_app=content_graph.compile()
print("Content Classifier Ready")



Content Classifier Ready


In [21]:
# Test 1: Positive content (should be approved)
print("=" * 60)
print("TEST 1: Positive Content")
print("=" * 60)

result=content_app.invoke({
    "text":"This is a great and amazing product! Excellent quality and very helpful.",
    "score":0.0,
    "message":"",
    "category":""
    
})

print(f"\nMessage: {result['category'].upper()}")
print(f"Message: {result['message'].upper()}")

TEST 1: Positive Content
Analyzing Content : This is a great and amazing product! Excellent qua....
Content Score:1.00
Routing to approve
✅ Content APPROVED

Message: APPROVED
Message: CONTENT APPROVED FOR PUBLICATION
